In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/bismazahoor/mysmarts/my_smarts_selection.json


# Env code, DON"T TOUCH

In [ ]:
!git clone https://github.com/bismazahoor27/try_tok.git


Cloning into 'Fragsmiles_ours'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 46 (delta 4), reused 46 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 1.94 MiB | 13.97 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
import subprocess, sys, shutil, os, types, site
from pathlib import Path

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-500:])
    if result.returncode != 0:
        print("ERR:", result.stderr[-300:])
    return result.returncode

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Pin numpy FIRST
# ─────────────────────────────────────────────────────────────────────────────
print("Installing numpy==2.0.2 (pinned) ...")
run("pip install numpy==2.0.2 --force-reinstall --quiet")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Install pomegranate BEFORE molsets
# ─────────────────────────────────────────────────────────────────────────────
print("Installing pomegranate>=1.0.4 ...")
run("pip install 'pomegranate>=1.0.4' --quiet")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Install molsets (no-deps) + remaining deps
# ─────────────────────────────────────────────────────────────────────────────
print("Installing molsets (no-deps) ...")
run("pip install molsets --no-deps --quiet")

print("Installing molsets remaining deps ...")
run("pip install scipy==1.13.0 fcd-torch tqdm matplotlib seaborn six rdkit --quiet")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Patch: rdkit.six shim
# ─────────────────────────────────────────────────────────────────────────────
import six
shim = types.ModuleType("rdkit.six")
shim.__dict__.update(six.__dict__)
sys.modules["rdkit.six"] = shim
print("rdkit.six shim applied ✓")

startup_dir = Path.home() / ".ipython/profile_default/startup"
startup_dir.mkdir(parents=True, exist_ok=True)
(startup_dir / "rdkit_six_shim.py").write_text("""
import sys, types
try:
    import six
    if "rdkit.six" not in sys.modules:
        shim = types.ModuleType("rdkit.six")
        shim.__dict__.update(six.__dict__)
        sys.modules["rdkit.six"] = shim
except ImportError:
    pass
""")
print("rdkit.six shim persisted to IPython startup ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Patch: moses/metrics/utils.py DataFrame.append removal
# ─────────────────────────────────────────────────────────────────────────────
UTILS_PATH = '/usr/local/lib/python3.12/dist-packages/moses/metrics/utils.py'
with open(UTILS_PATH, 'r') as f:
    content = f.read()
old = "_mcf.append(_pains, sort=True)['smarts'].values"
new = "pd.concat([_mcf, _pains], sort=True)['smarts'].values"
if old in content:
    content = content.replace(old, new)
    with open(UTILS_PATH, 'w') as f:
        f.write(content)
    print("moses utils.py patched ✓")
else:
    print("moses utils.py already patched ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Patch: pandas.append compatibility shim
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
if not hasattr(pd.DataFrame, 'append'):
    def _append(self, other, ignore_index=False, **kwargs):
        other_df = other if isinstance(other, pd.DataFrame) else pd.DataFrame([other])
        return pd.concat([self, other_df], ignore_index=ignore_index)
    pd.DataFrame.append = _append
    print("pandas.append shim applied ✓")
else:
    print("pandas.append already present ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — Import moses
# ─────────────────────────────────────────────────────────────────────────────
for key in list(sys.modules.keys()):
    if 'moses' in key:
        del sys.modules[key]

import moses
print(f"moses {moses.__version__} imported ✓  ({os.path.dirname(moses.__file__)})")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8 — Install r-fragSMILES from try_tok repo (no SMARTS files needed)
# ─────────────────────────────────────────────────────────────────────────────
REPO_DIR = '/kaggle/working/try_tok'   # root of cloned repo (has setup.py)

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', REPO_DIR, '--no-build-isolation', '--quiet'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip install failed — falling back to sys.path insertion")
    sys.path.insert(0, REPO_DIR)
else:
    print("r-fragSMILES (try_tok) installed via pip ✓")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9 — Sanity check: round-trip encode/decode (no smarts_path needed)
# ─────────────────────────────────────────────────────────────────────────────
from chemicalgof import encode, decode, split
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

_frag = encode('c1ccccc1')
_back = decode(_frag)
assert Chem.CanonSmiles('c1ccccc1') == Chem.CanonSmiles(_back), "Round-trip failed!"
print(f"Benzene r-fragSMILES: {_frag}")
print(f"Tokens:               {split(_frag)}")
print(f"Round-trip OK ✓")

print("\n=== Environment ready ===")


Installing numpy==2.0.2 (pinned) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 89.2 MB/s eta 0:00:00

Installing pomegranate>=1.0.4 ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.5 MB/s eta 0:00:00

Installing molsets (no-deps) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 35.5 MB/s eta 0:00:00

Installing molsets remaining deps ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 57.8 MB/s eta 0:00:00

rdkit.six shim applied ✓
rdkit.six shim persisted to IPython startup ✓
moses utils.py patched ✓
pandas.append shim applied ✓
moses 0.3.1 impor

In [4]:
print("Loading MOSES datasets...")
train_smiles = list(moses.get_dataset('train'))
test_smiles  = list(moses.get_dataset('test'))
test_sf      = list(moses.get_dataset('test_scaffolds'))

print(f"Train:          {len(train_smiles):,}")
print(f"Test:           {len(test_smiles):,}")
print(f"Test scaffolds: {len(test_sf):,}")
print(f"Sample:         {train_smiles[:2]}")

Loading MOSES datasets...
Train:          1,584,663
Test:           176,074
Test scaffolds: 176,225
Sample:         ['CCCS(=O)c1ccc2[nH]c(=NC(=O)OC)[nH]c2c1', 'CC(C)(C)C(=O)C(Oc1ccc(Cl)cc1)n1ccnc1']


In [ ]:
import sys, os, gc, time
for key in list(sys.modules.keys()):
    if 'chemicalgof' in key:
        del sys.modules[key]

import moses
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
from chemicalgof import encode, decode, split
from chemicalgof.bpe import BPETrainer, BPETokenizer

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CORES = os.cpu_count()

print(f"Device:    {device}")
print(f"CPU cores: {NUM_CORES}")

# Sanity test (no smarts_path needed)
_frag = encode('c1ccccc1')
_back = decode(_frag)
print(f"Benzene r-fragSMILES: {_frag}")
print(f"Tokens:               {split(_frag)}")
print(f"Round-trip OK:        {Chem.CanonSmiles('c1ccccc1') == Chem.CanonSmiles(_back)} ✓")


Device:            cuda
CPU cores:         4
SMARTS exists:     True
Benzene fragSMILES: c1ccccc1
Round-trip OK:      True ✓


# Env code done!

# Code for our fragsmiles starts

In [ ]:
# r-fragSMILES benchmark — no fragment_matching.py patching required
import time
from chemicalgof import encode, split
from chemicalgof.gof import classify_fragment_type

# Verify node_type classification
print("Fragment type classification:")
for smi, expected in [('c1ccccc1', 'ring'), ('CC', 'linker'), ('C', 'sidechain'),
                       ('c1ccc2ncccc2c1', 'ring'), ('CCN', 'linker')]:
    t = classify_fragment_type(smi)
    print(f"  {smi:20s} → {t:10s}  (expected {expected})")

# Sample encodings from the training set
print("\nSample r-fragSMILES encodings:")
for smi in train_smiles[:5]:
    fs   = encode(smi)
    toks = split(fs)
    print(f"  {smi[:45]:45s} → {toks}")

# Benchmark: 500 molecules
t0 = time.time()
ok = 0
for smi in train_smiles[:500]:
    try:
        encode(smi)
        ok += 1
    except Exception:
        pass
elapsed = time.time() - t0
per_mol = elapsed / 500
eta_hrs = per_mol * len(train_smiles) / NUM_CORES / 3600

print(f"\nBenchmark (500 mols): {ok}/500 OK")
print(f"  Per molecule:                    {per_mol*1000:.2f} ms")
print(f"  ETA ({NUM_CORES} cores, {len(train_smiles):,} mols): {eta_hrs:.1f} hrs")


Patching: /usr/local/lib/python3.12/dist-packages/chemicalgof/fragment_matching.py
File patched ✓
Module reloaded ✓
Warmup done - patterns compiled ✓

Per molecule:         6.18 ms
ETA (4 cores 1.58M):  0.7 hrs


In [ ]:
# BPE vocabulary preview on a small subset
# Full BPE training runs after the parallel encoding step.
import time
from chemicalgof import encode, split
from chemicalgof.bpe import BPETrainer, BPETokenizer
from collections import Counter

PREVIEW_N = 5000   # molecules for the preview

print(f"Building BPE preview on {PREVIEW_N:,} molecules ...")
t0 = time.time()

preview_tokens = []
for smi in train_smiles[:PREVIEW_N]:
    try:
        preview_tokens.append(split(encode(smi)))
    except Exception:
        pass

trainer = BPETrainer()
trainer.fit(preview_tokens, max_merges=200, min_freq=5)

print(f"  Encoded {len(preview_tokens):,} molecules in {time.time()-t0:.1f}s")
print(f"  BPE merges learned: {len(trainer.merges)}")

# Show the top-20 most impactful merges (connector + ring pairs)
print("\nTop 20 BPE merges (connector→ring pairs get priority):")
for i, (a, b) in enumerate(trainer.merges[:20]):
    print(f"  {i+1:2d}.  {a!r:30s} + {b!r}")

# Base-token frequency before BPE
all_base = [t for seq in preview_tokens for t in seq]
base_vocab_size = len(set(all_base))

# Merged token frequency after BPE
tok = BPETokenizer(trainer)
merged_seqs   = [tok.encode(seq) for seq in preview_tokens]
all_merged    = [t for seq in merged_seqs for t in seq]
merged_vocab  = len(set(all_merged))
token_counts  = Counter(len(seq) for seq in merged_seqs)
avg_len_base  = sum(len(s) for s in preview_tokens) / len(preview_tokens)
avg_len_merged= sum(len(s) for s in merged_seqs)    / len(merged_seqs)

print(f"\nVocab size:   {base_vocab_size:,} base  →  {merged_vocab:,} after BPE (200 merges, min_freq=5)")
print(f"Avg seq len:  {avg_len_base:.1f} base tokens  →  {avg_len_merged:.1f} BPE tokens")
print(f"Token reduction: {(1 - avg_len_merged/avg_len_base)*100:.1f}%")


Compiled: 0/882 patterns in 0.0s
Failed:   119
Time per molecule (882 patterns): 0.1ms
ETA (4 cores, 1.58M): 0.0 hrs


In [8]:
import os
print(os.cpu_count())

4


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: Parallel encode all training SMILES → base fragSMILES token lists
# ─────────────────────────────────────────────────────────────────────────────
import json, re, time, os
import numpy as np
from collections import Counter
from joblib import Parallel, delayed
from tqdm import tqdm

CHECKPOINT_BASE   = '/kaggle/working/checkpoint_base_tokens.npy'
BPE_MERGES_PATH   = '/kaggle/working/bpe_merges.json'
VOCAB_PATH        = '/kaggle/working/vocab.json'
TOKENS_OUT        = '/kaggle/working/train_tokens.npy'

MAX_LEN = 100
N_JOBS  = 4
CHUNK   = 200_000

# BPE settings (r-fragSMILES design: cap at 2048, min_freq 50)
BPE_MAX_MERGES = 2048
BPE_MIN_FREQ   = 50

# ── Worker: encode SMILES → base fragSMILES token list ───────────────────────
def encode_to_base_tokens(smi: str):
    try:
        from chemicalgof import encode, split
        return split(encode(smi))
    except Exception:
        return None


# ── Phase 1: Resume from checkpoint ──────────────────────────────────────────
if os.path.exists(CHECKPOINT_BASE):
    base_tokens = list(np.load(CHECKPOINT_BASE, allow_pickle=True))
    remaining   = train_smiles[len(base_tokens):]
    print(f"Resuming from {len(base_tokens):,} / {len(train_smiles):,}")
else:
    base_tokens = []
    remaining   = train_smiles
    print(f"Starting fresh — {len(remaining):,} molecules")

t0      = time.time()
total_c = (len(remaining) + CHUNK - 1) // CHUNK

for i in range(0, len(remaining), CHUNK):
    chunk     = remaining[i : i + CHUNK]
    chunk_num = i // CHUNK + 1

    results = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(encode_to_base_tokens)(smi)
        for smi in tqdm(chunk, desc=f'Chunk {chunk_num}/{total_c}', leave=False)
    )
    base_tokens.extend([r for r in results if r is not None])
    np.save(CHECKPOINT_BASE, np.array(base_tokens, dtype=object))

    done = len(base_tokens)
    eta  = ((time.time() - t0) / done) * (len(remaining) - done + 1) if done > 0 else 0
    print(f"Chunk {chunk_num}/{total_c} | encoded: {done:,} | ETA: {eta/3600:.2f} hrs")

print(f"\nPhase 1 done: {len(base_tokens):,}/{len(train_smiles):,} molecules encoded ✓")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: Train BPE on base token corpus (ring+connector merges only)
# ─────────────────────────────────────────────────────────────────────────────
from chemicalgof.bpe import BPETrainer, BPETokenizer

print(f"\nTraining BPE (max_merges={BPE_MAX_MERGES}, min_freq={BPE_MIN_FREQ}) ...")
t_bpe = time.time()

trainer = BPETrainer()
trainer.fit(base_tokens, max_merges=BPE_MAX_MERGES, min_freq=BPE_MIN_FREQ)
trainer.save(BPE_MERGES_PATH)

print(f"BPE merges learned: {len(trainer.merges):,}  (saved to {BPE_MERGES_PATH}) ✓")
print(f"BPE training time:  {time.time()-t_bpe:.1f}s")

tokenizer = BPETokenizer(trainer)

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: Apply BPE → build vocabulary → index into int32 array
# ─────────────────────────────────────────────────────────────────────────────
print("\nApplying BPE merges to corpus ...")
bpe_tokens = [tokenizer.encode(seq) for seq in tqdm(base_tokens, desc='BPE encode')]

# Build vocabulary from BPE-merged tokens
all_toks     = [tok for seq in bpe_tokens for tok in seq]
token_counts = Counter(all_toks)

SPECIAL   = ['<pad>', '<sos>', '<eos>', '<unk>']
vocab     = SPECIAL + [t for t, _ in token_counts.most_common()]
token2idx = {t: i for i, t in enumerate(vocab)}
idx2token = {i: t for t, i in token2idx.items()}

with open(VOCAB_PATH, 'w') as f:
    json.dump(token2idx, f)

avg_len_base = sum(len(s) for s in base_tokens) / len(base_tokens)
avg_len_bpe  = sum(len(s) for s in bpe_tokens)  / len(bpe_tokens)
print(f"Vocab size:      {len(vocab):,}  (saved to {VOCAB_PATH}) ✓")
print(f"Avg seq length:  {avg_len_base:.1f} base → {avg_len_bpe:.1f} after BPE")
print(f"Token reduction: {(1 - avg_len_bpe/avg_len_base)*100:.1f}%")

# Encode token strings → padded int32 array with SOS/EOS
print("Indexing BPE tokens into int array ...")
PAD = token2idx['<pad>']
SOS = token2idx['<sos>']
EOS = token2idx['<eos>']
UNK = token2idx['<unk>']

rows = np.full((len(bpe_tokens), MAX_LEN), PAD, dtype=np.int32)

for idx, toks in enumerate(tqdm(bpe_tokens, desc='Indexing')):
    ids = [SOS] + [token2idx.get(t, UNK) for t in toks] + [EOS]
    ids = ids[:MAX_LEN]
    rows[idx, :len(ids)] = ids

np.save(TOKENS_OUT, rows)
print(f"Saved {rows.shape} int32 array → {TOKENS_OUT} ✓")
print(f"\n=== DONE  {len(bpe_tokens):,}/{len(train_smiles):,} molecules ready for training ===")


Starting fresh — 1,584,663 molecules


Chunk 1/8:   0%|          | 0/200000 [00:00<?, ?it/s]
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 180, in <module>
    exitcode = process_obj._bootstrap()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._

Chunk 1/8 | 532s | Encoded: 199,791 | ETA: 1.02 hrs


Chunk 2/8 | 525s | Encoded: 399,744 | ETA: 0.87 hrs


Chunk 3/8 | 524s | Encoded: 599,622 | ETA: 0.72 hrs


...

Chunk 6/8 | 537s | Encoded: 1,199,529 | ETA: 0.28 hrs


Chunk 7/8 | 543s | Encoded: 1,399,516 | ETA: 0.14 hrs


Chunk 8/8 | 484s | Encoded: 1,584,126 | ETA: 0.00 hrs

Building vocabulary...
Vocab size: 4,762 — saved to /kaggle/working/vocab.json ✓
Indexing tokens into int array...


Indexing: 100%|██████████| 1584126/1584126 [00:05<00:00, 267115.51it/s]


Saved (1584126, 100) int32 array → /kaggle/working/train_tokens.npy ✓

=== DONE  1,584,126/1,584,663 molecules encoded ===


In [ ]:
# ── Drop-in Dataset (no re-tokenisation ever) ─────────────────────────────────
# Use this in place of FragSMILESDataset going forward:
import torch
from torch.utils.data import Dataset, DataLoader

MAX_LEN    = 100
BATCH_SIZE = 512
class FragSMILESDataset(Dataset):
    def __init__(self, tokens_path=TOKENS_OUT):
        self.data = torch.from_numpy(np.load(tokens_path))   # (N, MAX_LEN) int32
    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self, idx):
        return self.data[idx]   # ready-to-use LongTensor, zero string work

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

dataset = FragSMILESDataset()
loader  = DataLoader(
    dataset,
    batch_size      = BATCH_SIZE,
    shuffle         = True,
    num_workers     = 0, # Note: Since __getitem__ is now instantaneous, you might be able to drop this to 0
    pin_memory      = True,
    drop_last       = True,   # prevents uneven last-batch crash with DataParallel
)

print(f"Dataset size:       {len(dataset):,}")
print(f"Vocab size:         {len(vocab):,}") # Assuming vocab is defined in your environment
print(f"Sample batch shape: {next(iter(loader)).shape} ✓")


Device: cuda
Dataset size:       1,584,126
Vocab size:         4,762
Sample batch shape: torch.Size([512, 100]) ✓


In [11]:
import torch.nn as nn

class FragSMILES_RNN(nn.Module):
    def __init__(self, vocab_size, embed_size=256, hidden_size=512,
                 num_layers=3, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn       = nn.LSTM(
            embed_size, hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout
        )
        self.fc      = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        emb         = self.dropout(self.embedding(x))
        out, hidden = self.rnn(emb, hidden)
        logits      = self.fc(self.dropout(out))
        return logits, hidden

model = FragSMILES_RNN(vocab_size=len(vocab)).to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using {torch.cuda.device_count()} GPUs ✓")

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device:     {device} ✓")

Using 2 GPUs ✓
Parameters: 9,441,434
Device:     cuda ✓


In [12]:
from tqdm.notebook import tqdm  # notebook version works better on Kaggle
import time, pandas as pd

N_EPOCHS  = 5
SAVE_PATH = '/kaggle/working/rnn_best.pt'

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)
scaler    = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

best_loss = float('inf')
history   = []

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0.0
    t_start    = time.time()

    for i, batch in enumerate(loader):
        # batch = batch.to(device, non_blocking=True)
        batch = batch.to(device, dtype=torch.long, non_blocking=True)
        x, y  = batch[:, :-1], batch[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        if scaler:
            with torch.amp.autocast('cuda'):
                logits, _ = model(x)
                loss = criterion(logits.reshape(-1, len(vocab)), y.reshape(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, _ = model(x)
            loss = criterion(logits.reshape(-1, len(vocab)), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()

        # Print every 100 batches
        if i % 1000 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(loader)} | Loss: {loss.item():.4f} | Time: {time.time()-t_start:.0f}s", flush=True)

    avg_loss   = total_loss / len(loader)
    elapsed    = time.time() - t_start
    current_lr = optimizer.param_groups[0]['lr']
    history.append({'epoch': epoch+1, 'loss': avg_loss})
    scheduler.step(avg_loss)

    print(f"\n{'='*50}", flush=True)
    print(f"Epoch {epoch+1:02d}/{N_EPOCHS} DONE | Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s", flush=True)
    print(f"{'='*50}\n", flush=True)

    if avg_loss < best_loss:
        best_loss = avg_loss
        m = model.module if hasattr(model, 'module') else model
        torch.save({
            'model_state': m.state_dict(),
            'vocab':       vocab,
            'token2idx':   token2idx,
            'loss':        best_loss,
        }, SAVE_PATH)
        print(f"Saved best model ✓", flush=True)

print(f"Training complete! Best loss: {best_loss:.4f} ✓")
pd.DataFrame(history).to_csv('/kaggle/working/loss_history.csv', index=False)

Epoch 1 | Batch 0/3094 | Loss: 8.4653 | Time: 2s
Epoch 1 | Batch 1000/3094 | Loss: 1.3488 | Time: 173s
Epoch 1 | Batch 2000/3094 | Loss: 1.1404 | Time: 349s
Epoch 1 | Batch 3000/3094 | Loss: 1.0507 | Time: 526s

Epoch 01/5 DONE | Loss: 1.3777 | Time: 542.3s

Saved best model ✓
Epoch 2 | Batch 0/3094 | Loss: 1.0893 | Time: 0s
Epoch 2 | Batch 1000/3094 | Loss: 1.0154 | Time: 177s
Epoch 2 | Batch 2000/3094 | Loss: 0.9919 | Time: 353s
Epoch 2 | Batch 3000/3094 | Loss: 0.9896 | Time: 530s

Epoch 02/5 DONE | Loss: 1.0114 | Time: 546.6s

Saved best model ✓
Epoch 3 | Batch 0/3094 | Loss: 0.9349 | Time: 0s
Epoch 3 | Batch 1000/3094 | Loss: 0.9674 | Time: 177s
Epoch 3 | Batch 2000/3094 | Loss: 0.9573 | Time: 354s
Epoch 3 | Batch 3000/3094 | Loss: 0.9707 | Time: 530s

Epoch 03/5 DONE | Loss: 0.9639 | Time: 547.0s

Saved best model ✓
Epoch 4 | Batch 0/3094 | Loss: 0.9472 | Time: 0s
Epoch 4 | Batch 1000/3094 | Loss: 0.9521 | Time: 177s
Epoch 4 | Batch 2000/3094 | Loss: 0.9269 | Time: 354s
Epoch 4 |

In [13]:
pd.DataFrame(history).to_csv('/kaggle/working/loss_history.csv', index=False)

In [19]:
# Sample 30,000 molecules
import torch.nn.functional as F
from tqdm.notebook import tqdm

# Load best model
ckpt      = torch.load('/kaggle/working/rnn_best.pt', map_location=device)
vocab     = ckpt['vocab']
token2idx = ckpt['token2idx']
idx2token = {i: t for i, t in enumerate(vocab)}

sample_model = FragSMILES_RNN(vocab_size=len(vocab)).to(device)
sample_model.load_state_dict(ckpt['model_state'])
sample_model.eval()

N_SAMPLES = 30000
SOS_IDX   = token2idx['<sos>']
all_seqs  = []

print(f"Sampling {N_SAMPLES:,} molecules...")
with torch.no_grad():
    while len(all_seqs) < N_SAMPLES:
        cur_bs    = min(512, N_SAMPLES - len(all_seqs))
        x         = torch.full((cur_bs, 1), SOS_IDX, dtype=torch.long, device=device)
        hidden    = None
        sequences = [[] for _ in range(cur_bs)]
        done      = [False] * cur_bs

        for _ in range(MAX_LEN):
            logits, hidden = sample_model(x, hidden)
            probs  = F.softmax(logits[:, -1, :], dim=-1)
            next_t = torch.multinomial(probs, 1)
            for i in range(cur_bs):
                if not done[i]:
                    tok = idx2token[next_t[i].item()]
                    if tok == '<eos>':
                        done[i] = True
                    elif tok not in ('<pad>', '<sos>'):
                        sequences[i].append(tok)
            x = next_t
            if all(done):
                break

        all_seqs.extend(sequences)

print(f"Sampled {len(all_seqs):,} ✓")
print(f"Sample sequence: {all_seqs[0]}")

Sampling 30,000 molecules...
Sampled 30,000 ✓
Sample sequence: ['<1>', 'c1cn[nH]c1', '<0>', '<3>', '(', 'C', ')', '<4>', '(', 'C', ')', 'C', '<0,d>', '(', 'O', ')', '<8>']


In [ ]:
# Decode BPE token sequences → SMILES (parallel)
# Pipeline: BPE-merged tokens → base fragSMILES tokens → decode() → SMILES
from joblib import Parallel, delayed

BPE_MERGES_PATH = '/kaggle/working/bpe_merges.json'

def decode_worker(token_list):
    try:
        from chemicalgof import decode
        from chemicalgof.bpe import BPETokenizer
        from rdkit import Chem

        # Load BPE tokenizer once per worker process
        global _BPE_TOK
        try:
            _BPE_TOK
        except NameError:
            _BPE_TOK = BPETokenizer.load(BPE_MERGES_PATH)

        # Step 1: BPE decode → base fragSMILES tokens
        base_tokens = _BPE_TOK.decode(token_list)

        # Step 2: fragSMILES tokens → SMILES (strict_chirality=False drops
        #         spurious stereo tags instead of raising InvalidChirality)
        smi = decode(base_tokens, strict_chirality=False)
        mol = Chem.MolFromSmiles(smi)
        return Chem.MolToSmiles(mol) if mol else None
    except Exception:
        return None


print("Decoding BPE token sequences → SMILES ...")
decoded = Parallel(n_jobs=4, backend='loky')(
    delayed(decode_worker)(seq) for seq in tqdm(all_seqs)
)

valid_smiles = [s for s in decoded if s is not None]
print(f"Valid: {len(valid_smiles):,}/{len(decoded):,} ({len(valid_smiles)/len(decoded)*100:.1f}%)")

with open('/kaggle/working/sampled_smiles.txt', 'w') as f:
    for smi in valid_smiles:
        f.write(smi + '\n')
print("Saved /kaggle/working/sampled_smiles.txt ✓")


Decoding to SMILES...


  0%|          | 0/30000 [00:00<?, ?it/s]


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/backend/popen_loky_posix.py", line 180, in <module>
    exitcode = process_obj._bootstrap()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packag

Valid: 17,452/30,000 (58.2%)
Saved ✓


In [21]:
# Verify + load everything
import numpy as np
print(f"numpy: {np.__version__}")  # should be 2.0.2

from moses.metrics import get_all_metrics
import moses, torch, pandas as pd

# Load saved valid smiles (already generated!)
with open('/kaggle/working/sampled_smiles.txt', 'r') as f:
    valid_smiles = [line.strip() for line in f if line.strip()]
print(f"Loaded: {len(valid_smiles):,} valid SMILES ✓")

train_smiles = list(moses.get_dataset('train'))
test_smiles  = list(moses.get_dataset('test'))
test_sf      = list(moses.get_dataset('test_scaffolds'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Computing MOSES metrics...")
metrics = get_all_metrics(
    gen            = valid_smiles,
    test           = test_smiles,
    test_scaffolds = test_sf,
    train          = train_smiles,
    n_jobs         = 4,
    device         = str(device)
)

# Print results
print("\n=== Results ===")
for k, v in metrics.items():
    print(f"  {k:20s}: {round(v,4) if isinstance(v,float) else v}")

numpy: 2.0.2
Loaded: 17,442 valid SMILES ✓
Computing MOSES metrics...

=== Results ===
  valid               : 1.0
  unique@1000         : 0.996
  unique@10000        : 0.979
  FCD/Test            : 4.3685
  SNN/Test            : 0.5326
  Frag/Test           : 0.964
  Scaf/Test           : 0.8499
  FCD/TestSF          : 4.8655
  SNN/TestSF          : 0.5054
  Frag/TestSF         : 0.965
  Scaf/TestSF         : 0.1176
  IntDiv              : 0.8604
  IntDiv2             : 0.8523
  Filters             : 0.8476
  logP                : 0.6626
  SA                  : 0.0879
  QED                 : 0.0437
  weight              : 60.2264
  Novelty             : 0.9745
